# 01-PyTorch Basics

In [ ]:
import torch

## Tensors

In [ ]:
t1 = torch.tensor(4.)
t1

In [ ]:
t1.dtype

In [ ]:
t2 = torch.tensor([1.,2,3,4])
t2

In [ ]:
t2.dtype

In [ ]:
t3 = torch.tensor([
    [5.,6.],
    [7,8],
    [9,10]
])
t3

In [ ]:
t3.shape

In [ ]:
# 3d tensor
t4 = torch.tensor([
    [
        [11.,12,13],
        [24,25,26]
    ],
    [
        [1.,2,3],
        [4,5,6]
    ]
])
t4

In [ ]:
t4.shape

In [ ]:
t5 = torch.tensor([
    [1.,2,3,6],
    [4,5]
])
t5.shape

## Tensor Operations and Gradients

In [ ]:
x = torch.tensor(3.)
w = torch.tensor(4., requires_grad=True)
b = torch.tensor(5., requires_grad=True)
x,w,b

In [ ]:
y = w*x + b
y

In [ ]:
# compute derivatives
y.backward()

# display gradients
print('dy/dx: ', x.grad)
print('dy/dw: ', w.grad)
print('dy/db: ', b.grad)

## Tensor Functions

In [ ]:
t6 = torch.full((3,2), 42)
t6

In [ ]:
t3

In [ ]:
t7 = torch.cat((t3,t6))
t7

In [ ]:
t8 = torch.sin(t7)
t8

In [ ]:
t9 = t8.reshape(3,2,2)
t9

## Numpy Interoperatibility

In [ ]:
import numpy as np

In [ ]:
x = np.array([
    [1,2],
    [3,4]
])

x

In [ ]:
y = torch.from_numpy(x)
y

In [ ]:
x.dtype, y.dtype

In [ ]:
z = y.numpy()
z

# 02-Linear Regression

In [ ]:
import numpy as np
import torch

In [ ]:
inputs = np.array([
    [73,67,43],
    [91,88,64],
    [87,134,58],
    [102,43,37],
    [69,96,70]
], dtype='float32')

In [ ]:
targets = np.array([
    [56,70],
    [81, 101],
    [119, 133],
    [22, 37],
    [103, 119]
], dtype='float32')

In [ ]:
inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)
inputs, targets

In [ ]:
# weights and biases
w = torch.randn(2,3, requires_grad=True)
b = torch.randn(2, requires_grad=True)
w,b

In [ ]:
def model(x):
    return x @ w.t() + b

In [ ]:
inputs @ w.t() + b

In [ ]:
preds = model(inputs)
print(preds)
print(targets)

## Loss Function

In [ ]:
# mean squared error
diff = preds - targets
torch.sum(diff * diff) / diff.numel()

In [ ]:
def mse(t1,t2):
    diff = t1-t2
    return torch.sum(diff * diff) / diff.numel()

In [ ]:
loss = mse(preds, targets)
loss

### Gradient Computation

In [ ]:
# compute gradients
loss.backward()

In [ ]:
print(w)
print(w.grad)

In [ ]:
print(b, b.grad, sep="\n")

In [ ]:
with torch.no_grad():
    w -= w.grad * 1e-5
    b -= b.grad * 1e-5

In [ ]:
w,b

In [ ]:
preds = model(inputs)
loss = mse(preds, targets)
loss


In [ ]:
# reset gradients to zero 
w.grad.zero_()
b.grad.zero_()
w.grad, b.grad

In [ ]:
# Training for mutiple epochs
for i in range(100):
    preds = model(inputs)
    loss = mse(preds, targets)
    loss.backward()

    with torch.no_grad():
        w -= w.grad * 1e-5
        b -= b.grad * 1e-5
        w.grad.zero_()
        b.grad.zero_()

In [ ]:
preds = model(inputs)
loss = mse(preds, targets)
loss

In [ ]:
preds

In [ ]:
targets

## Using PyTorch

In [ ]:
# Input (temp, rainfall, humidity)
inputs = np.array([[73, 67, 43], 
                   [91, 88, 64], 
                   [87, 134, 58], 
                   [102, 43, 37], 
                   [69, 96, 70], 
                   [74, 66, 43], 
                   [91, 87, 65], 
                   [88, 134, 59], 
                   [101, 44, 37], 
                   [68, 96, 71], 
                   [73, 66, 44], 
                   [92, 87, 64], 
                   [87, 135, 57], 
                   [103, 43, 36], 
                   [68, 97, 70]], 
                  dtype='float32')

# Targets (apples, oranges)
targets = np.array([[56, 70], 
                    [81, 101], 
                    [119, 133], 
                    [22, 37], 
                    [103, 119],
                    [57, 69], 
                    [80, 102], 
                    [118, 132], 
                    [21, 38], 
                    [104, 118], 
                    [57, 69], 
                    [82, 100], 
                    [118, 134], 
                    [20, 38], 
                    [102, 120]], 
                   dtype='float32')

inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)

### Dataset and DataLoader

In [ ]:
from torch.utils.data import TensorDataset

In [ ]:
train_ds = TensorDataset(inputs, targets)
train_ds[0:3]

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
batch_size = 5
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [ ]:
for xb, yb in train_dl:
    print(xb)
    print(yb)
    break

In [ ]:
# defining model
model = torch.nn.Linear(3,2)
print(model.weight)
print(model.bias)

In [ ]:
# Parameters
list(model.parameters())

In [ ]:
# predictions
preds = model(inputs)
preds

### Loss Function

In [ ]:
# Loss function
import torch.nn.functional as F

In [ ]:
loss_fn = F.mse_loss

In [ ]:
loss = loss_fn(model(inputs), targets)
loss

### Optimizers

In [ ]:
# Optimizers
opt = torch.optim.SGD(model.parameters(), lr=1e-5)

### Model Training

In [ ]:
def fit(num_epochs, model, loss_fn, opt, train_dl):
    for epochs in range(num_epochs):
        for xb, yb in train_dl:
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()
        
        if (epochs+1) % 10 == 0:
            print("Epoch [{}/{}], Loss: {:.4f}".format(epochs+1, num_epochs, loss.item()))


In [ ]:
fit(100, model, loss_fn, opt, train_dl)

In [ ]:
preds = model(inputs)
preds

In [ ]:
targets

In [ ]:
model(torch.tensor([[75., 63, 44]]))